# 第69课 · 把指数级路径装进一张表——CTC 前向算法（forward algorithm），纯 NumPy 实现

**学习目标**
- 理解 CTC 的核心挑战：不知道字符在哪一帧对齐（alignment）
- 掌握前向变量（forward variable，alpha） α 的递推定义
- 用纯 NumPy 实现 CTC 前向算法并数值验证

> 与 L68 **同一套记号**；数值常走 log 域防下溢。

← **上一课**　[L68 · CTC 对齐原理](L68_ctc_alignment.ipynb)

> 上节课学习了 **CTC 对齐原理**：blank 符号、单调路径与标签折叠。  
> 本课将探讨 **CTC 前向算法**。

## 本课剧情：为什么朴素的路径枚举会"爆炸"？

上节课学了 CTC 的直觉：blank 符号让一个目标序列对应指数级多条路径。

但问题来了：**要是想精确算出 P("ab" | 6帧)，我们需要枚举 3⁶=729 条路径，逐一压缩验证，再相加**。T=100帧时就是 3¹⁰⁰条路径——彻底不可行。

**前向算法（Forward Algorithm）**的解法：用动态规划，把"枚举→求和"变成"递推→合并"。

**关键设计**：把目标序列 `["a","b"]` 扩展为包含 blank 的序列 `l' = [∅, a, ∅, b, ∅]`（每个字符前后插 blank），长度 `S = 2L+1`。

**前向变量** α[t][s] = "在第 t 帧结束时，以 l'[s] 结束的所有合法路径的概率和"。

**递推关系（三种情况）**：
```
情况1：l'[s] = blank，或与前2步字符相同
   α[t][s] = (α[t-1][s] + α[t-1][s-1]) × P(l'[s] | t)

情况2：l'[s] = 字符，且 l'[s] ≠ l'[s-2]
   α[t][s] = (α[t-1][s] + α[t-1][s-1] + α[t-1][s-2]) × P(l'[s] | t)
```

最终：`P(label | logits) = α[T-1][S-1] + α[T-1][S-2]`——最后一帧可以停在末尾 blank 或最后一个字符；log 域实现时这一步加法用 `logaddexp` 完成。

复杂度从指数 → **O(T·S)**，其中 `S = 2L + 1`；训练 ASR 成为可能。

### 什么是"合法路径"？

想象你在爬一座阶梯。每一帧是一级台阶，每个位置是台阶的"高度"。一条**合法路径**就是一种爬法：
- **从最低处开始**：第 0 帧必须从位置 0（blank）或位置 1（第一个字符）开始。
- **向上爬或停留**：每一步最多往上爬一级（即位置 s → s 或 s+1），或者**跳过一个台阶**（位置 s → s+2），但只在特定情况下允许。
- **为什么不能跳 s+3？** 如果允许任意大跳跃，那么"一个标签 a 对应的不同路径"就太多了，反而无法有效地用 DP 合并计算。关键约束是：**相邻两帧的位置最多差 2**，这样才能在 O(T·S) 复杂度内完成。

**为什么"跳两步"要额外检查 `l'[s] ≠ l'[s-2]`？**

这是为了避免**重复计算**。考虑这个例子：
- 目标标签是 `[a]`，扩展后 `l' = [∅, a, ∅]`（位置 0,1,2）
- 一条路径是：第 0 帧停在位置 0（∅），第 1 帧停在位置 1（a）
  - 这条路径的"跳跃方式"是从位置 0 跳到位置 1（跳 1 步）
- 另一条路径是：第 0 帧停在位置 1（a），第 1 帧停在位置 1（a）
  - 这条路径在位置 1 停留（跳 0 步，即停留）

如果我们**错误地允许从位置 -1 跳到位置 1**（跳 2 步），在这个系统中位置编号会混乱。但更重要的是，如果 `l'[s]` 和 `l'[s-2]` 都是**同一个字符**（比如都是 'a'），那么：
- 从 `l'[s-2]` 直接跳到 `l'[s]` 会"跳过中间的 blank"
- 但当路径折叠时，如果中间被跳过的 blank 本来是为了分隔两个相同字符用的，我们就会漏掉这个分隔，导致错误地把"a-blank-a"和"a-a"混为一谈

所以条件 `l'[s] ≠ l'[s-2]` 保证了：**只有当我们跳过的中间字符和目标字符不同时，才允许跳过中间的 blank**。这样折叠规则才能正确地工作。


In [ ]:
import numpy as np

In [ ]:
# 构造玩具示例：6 帧，词汇 {a:0, b:1, blank:2}，目标 'ab'
BLANK = 2
VOCAB_SIZE = 3
T = 6
LABELS = [0, 1]   # 'a', 'b'

np.random.seed(42)
logits = np.random.randn(T, VOCAB_SIZE)
# 转为 log 概率
# 数值稳定 log-softmax：先减最大值，避免 exp 溢出
logits_s = logits - logits.max(axis=1, keepdims=True)
log_probs = logits_s - np.log(np.exp(logits_s).sum(axis=1, keepdims=True))
print(f'log_probs shape: {log_probs.shape}  (T={T}, vocab={VOCAB_SIZE})')

## 🔬 先手填一张极小 α 表（T=3，标签 "AB"）

第一次做 CTC 的 DP，先别管 100 帧。用 **T=3、目标 `"AB"`** 手填三行 α，把递推"跑通"一次，代码就不吓人了。

**扩展标签** `l' = [_, A, _, B, _]`，位置 `s = 0..4`（`S = 2L+1 = 5`）。
`α[t][s]` = "到第 t 帧、且停在 `l'[s]` 上的所有合法路径概率和"。为看清结构，下面只标**哪些格非零**（`·` = 不可达，仍是 -inf）：

```
 s=      0(_)   1(A)   2(_)   3(B)   4(_)
t=0       ✓      ✓      ·      ·      ·     初始：只能从开头的 _ 或第一个字符 A 起步
t=1       ✓      ✓      ✓      ✓      ·     每步最多向右推进一格，所以 B(s=3) 此刻刚够得到
t=2       ·      ✓      ✓      ✓      ✓     3 帧要走完 "AB"，末尾必须落在 B(s=3) 或收尾 _(s=4)
```

**每格从哪几个前驱来**（就是 TODO 里的三路 `logaddexp`）：
- 停在原地：`α[t-1][s]`
- 从左一格推进：`α[t-1][s-1]`
- **跳过一个 blank**：`α[t-1][s-2]`，**仅当** `l'[s]` 是字符且 `l'[s] ≠ l'[s-2]`（避免把 `A_A` 错当成一个 A）

**最终答案** = `logaddexp(α[2][4], α[2][3])`——最后一帧可以停在收尾 `_` 或最后一个字符 `B`。

对照上表的"哪格可达"再去读下面的实现，三重 `if` 就是在描述"只能向右、最多跳过一个 blank"。


### 初始化和最后状态的直觉

**为什么第 0 帧只能从前两个位置起步？**

目标 `[a, b]` 扩展后变成 `[∅, a, ∅, b, ∅]`。在第 0 帧，我们还没发出任何字符，所以只能有两种开局方式：
1. **保持 blank**：停在位置 0（初始的∅），相当于"第 0 帧先打 blank，字符还没来"
2. **发出第一个字符**：停在位置 1（第一个 a），相当于"第 0 帧立即发出 a"

任何其他位置都意味着"一帧内同时跨过多个字符"，这违反了单调性。所以：
```
α[0, 0] = log P(∅ | frame_0)
α[0, 1] = log P(a | frame_0)
α[0, 2] = α[0, 3] = ... = -∞    # 不可达
```

**为什么最后一帧的答案是两个位置的 logaddexp？**

在第 T-1 帧（最后一帧），有两种合法的终态：
1. **停在最后一个字符**：位置 S-1（即最后的 b）
   - 这意味着"最后一帧发出了 b，然后路径结束"
2. **停在末尾 blank**：位置 S-2（即最后的∅）
   - 这意味着"字符已经在前面帧发出完了，最后一帧是 blank 填充"

这两种终态的概率都应该被累加，因为它们都能折叠到目标标签 `[a, b]`。位置 S-3（倒数第三个 b）及以前的位置不能作为终态，因为那样后续帧无法到达终点。

所以答案 = α[T-1][S-1] + α[T-1][S-2]（在 log 域用 logaddexp）。


### 扩展标签 l' 的具体构造与索引对应

**例子：原始标签 labels=[0, 1]（表示 'a'=0, 'b'=1），blank=2**

```
构造过程：
  1. 初始化：lprime = [blank] = [2]
  2. 循环遍历标签：
     - 加入 labels[0]=0：lprime = [2, 0]
     - 加入 blank：lprime = [2, 0, 2]
     - 加入 labels[1]=1：lprime = [2, 0, 2, 1]
     - 加入 blank：lprime = [2, 0, 2, 1, 2]

扩展标签：l' = [2, 0, 2, 1, 2]
                 ↓  ↓  ↓  ↓  ↓
                 ∅  a  ∅  b  ∅
位置索引：s=   0  1  2  3  4

长度 S = 2L+1 = 2*2+1 = 5
```

**当我们检查 `l'[s] != l'[s-2]` 时：**
- s=0: 没有 s-2，跳过
- s=1: 没有 s-2，跳过
- s=2: l'[2]=2（∅），l'[0]=2（∅） → 相等，不允许从 s-2 跳
- s=3: l'[3]=1（b），l'[1]=0（a） → 不等，**允许**从 s-2 跳
- s=4: l'[4]=2（∅），l'[2]=2（∅） → 相等，不允许从 s-2 跳

**关键观察**：这个条件确保了"只有当目标字符与两步前的字符不同时，才能跳过中间的 blank"。这样就不会把"a-blank-a"错误地合并成"a"。


## ✏️ 实现 CTC 前向算法

**签名**：
```python
def ctc_forward(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    # log_probs: (T, V) 对数概率矩阵
    # labels: list[int] 目标序列（不含 blank）
    # 返回: float  log P(labels | log_probs)
```

**5步实现路线**：

| 步骤 | 操作 | 关键点 |
|---|---|---|
| 1 | 构造扩展标签（extended label） `l'` | `[blank, L[0], blank, L[1], ..., blank]`，长度 `2L+1` |
| 2 | 初始化 α：`-inf`，设 `α[0][0]` 和 `α[0][1]` | 第0帧只能是 blank 或第1个字符 |
| 3 | 对 t=1..T-1：对 s=0..S-1 递推 | 情况1（blank/重复）取2项，情况2取3项 |
| 4 | 用 log-space 加法避免下溢 | `logsumexp(a, b) = a + log(1 + exp(b-a))` |
| 5 | 返回 `logsumexp(α[T-1][S-1], α[T-1][S-2])` | 两种结尾方式的概率和 |

**验收标准**：
- 与暴力枚举（`ctc_forward_brute_force`）结果误差 < 1e-5
- `T=6, L=["a","b"]` 时返回有限 float（不是 -inf）

**实现前先记一句**：`S = 2L + 1`，本课复杂度是 `O(T·S)`，不是 `O(T×2L)`。


### 实现中的边界细节

**下标检查**：
- `if s > 0`：确保 s-1 ≥ 0，允许从左一格跳来
- `if s > 1`：确保 s-2 ≥ 0，且满足"不同字符"条件，允许从左两格跳来

**为什么最后返回的是 alpha[-1, -1] 和 alpha[-1, -2]？**
- `alpha[-1, -1]` = `alpha[T-1, S-1]`：最后一帧、最后一个位置（对应目标标签的最后字符）
- `alpha[-1, -2]` = `alpha[T-1, S-2]`：最后一帧、倒数第二个位置（对应末尾的 blank）
- 这两种终态覆盖了所有合法路径的结尾方式：要么在字符上结束，要么在末尾 blank 上结束

**为什么不能只取 alpha[-1, -1]？**
- 有些合法路径会在目标标签完成后继续打 blank（比如前面帧就发出了所有字符）
- 这些路径的终态是 S-2（末尾 blank），概率不能丢弃

**逻辑流程图**（以 labels=[a,b], T=3 为例）：
```
t=0: [✓, ✓, ·, ·, ·]     初始：只有位置0,1可达
     ↓
t=1: [✓, ✓, ✓, ✓, ·]     递推：每帧最多推进1或2步
     ↓
t=2: [·, ✓, ✓, ✓, ✓]     最终：positions 3和4相加 = 答案
```


### Log 域等价（实现用此式）

```text
log α[t, s] = log p[t, l'[s]] + logaddexp(
    log α[t-1, s],      # 停留
    log α[t-1, s-1],    # 跳一步
    log α[t-1, s-2]     # 跳两步（仅当 l'[s] ≠ l'[s-2]）
)
```

`np.logaddexp(a, b)` = `log(exp(a) + exp(b))`，数值稳定；三路前驱可用两次 `logaddexp` 叠出来。

### 为什么要减最大值？下溢危机的具体例子

想象你手里有 100 个概率，每个都是 0.8，要把它们全乘起来：
```
P = 0.8^100
```

**直接计算会发生什么？**
- 在普通浮点数域：0.8^100 ≈ 2.04×10^(-10)，已经接近最小正浮点数（约 10^-308）
- 如果是 0.8^1000，结果就是 10^(-97)，溢出到 0，导致"下溢"——最后程序算出 P=0（完全错误）

**log 域的救星：**
```
log(P) = log(0.8^100) = 100 × log(0.8) ≈ 100 × (-0.223) = -22.3
```
不用管数值大小，只管"指数"——稳定得多。

**但还有个陷阱：exp 溢出**。如果 logits 本身很大（比如 [1000, 1001, 999]），直接算 softmax 会先算 exp：
```
softmax = exp([1000, 1001, 999]) / sum(exp(...))
```
exp(1000) 和 exp(1001) 都远超浮点数上限（约 10^308），直接爆炸！

**解决办法就是"减最大值"：**
```
logits_shifted = [1000, 1001, 999] - 1001 = [-1, 0, -2]
softmax = exp([-1, 0, -2]) / sum(exp(...))   # 数值都在可管理范围
log(softmax) = logits_shifted - log(sum(exp(logits_shifted)))
```
这样既避免了 exp 溢出，最后的数值结果也完全相同（因为概率的比值没变）。

**CTC 中的应用**：
- 输入 logits 可能很大（神经网络输出）→ 用 log-softmax 稳定
- 所有路径概率相乘 → 在 log 域变成求和 → 用 logaddexp 稳定相加
- 最后聚合多条路径 → logaddexp 一次搞定

下面的代码展示了"直接计算"和"log 稳定"的对比。


In [ ]:
# 数值演示：下溢与溢出的具体例子
import numpy as np

print("=" * 60)
print("【例子 1】多个概率相乘 → 下溢")
print("=" * 60)

# 模拟一条 100 帧的路径，每帧概率都是 0.8
T = 100
p_per_frame = 0.8
log_p_per_frame = np.log(p_per_frame)

# 直接在概率域相乘
try:
    p_direct = np.prod([p_per_frame] * T)
    print(f"直接计算：P = 0.8^{T} = {p_direct}")
    if p_direct == 0:
        print("  ⚠️  结果变成了 0（下溢到无穷小）！")
except:
    print("❌ 计算失败（溢出或下溢）")

# 在 log 域相加
log_p_log_domain = T * log_p_per_frame
print(f"\nlog 域计算：log(P) = {T} × {log_p_per_frame:.4f} = {log_p_log_domain:.4f}")
print(f"还原概率：P ≈ exp({log_p_log_domain:.4f}) ≈ {np.exp(log_p_log_domain):.2e}")
print("  ✅ 数值稳定，结果合理")

print("\n" + "=" * 60)
print("【例子 2】大 logits 的 softmax → exp 溢出")
print("=" * 60)

# 模拟神经网络的大输出值
logits = np.array([1000.0, 1001.0, 999.0])
print(f"Logits: {logits}")

# 错误的方式：直接 softmax
print("\n❌ 错误方式（直接 exp）：")
try:
    exp_logits = np.exp(logits)
    print(f"  exp(logits) = {exp_logits}")
    softmax_wrong = exp_logits / exp_logits.sum()
    print(f"  softmax = {softmax_wrong}")
    if np.any(np.isnan(softmax_wrong)) or np.any(np.isinf(softmax_wrong)):
        print("  ⚠️  出现 NaN 或 Inf（溢出）！")
except:
    print("  ❌ 计算失败（溢出）")

# 正确的方式：减最大值
print("\n✅ 正确方式（log-softmax 技巧）：")
m = logits.max()
logits_shifted = logits - m
print(f"  Logits - max = {logits_shifted}")
exp_shifted = np.exp(logits_shifted)
softmax_stable = exp_shifted / exp_shifted.sum()
print(f"  softmax = {softmax_stable}")
print(f"  log(softmax) = {np.log(softmax_stable)}")
print("  ✅ 数值稳定，没有溢出")

print("\n" + "=" * 60)
print("【例子 3】logaddexp 的稳定性")
print("=" * 60)

# 两个很小的 log 概率
log_p1 = -1000.0
log_p2 = -1001.0
print(f"log_p1 = {log_p1}, log_p2 = {log_p2}")

# 错误的方式：直接相加（在 log 域这是乘法，不是加法！）
print(f"\n❌ 错误方式（log_p1 + log_p2 = {log_p1 + log_p2}）")
print("  这实际上是在 log 域做了“乘法”，不是想要的“加法”")

# 正确的方式：logaddexp
log_sum = np.logaddexp(log_p1, log_p2)
print(f"\n✅ 正确方式（logaddexp）：")
print(f"  log(exp(log_p1) + exp(log_p2)) = {log_sum}")
print(f"  转换回概率：exp({log_sum}) ≈ {np.exp(log_sum):.2e}")
print("  ✅ 稳定计算了“加法”（在概率域）")

In [ ]:
a, b = np.log([0.3, 0.7])
assert np.isclose(np.logaddexp(a, b), np.log(1.0))
print("✅ logaddexp 演示：log(exp(a)+exp(b)) 在 log 域里能稳定相加")

### logaddexp 为什么需要：log 域里"加法"变成了什么？

**关键认识：log 是非线性的**

假设两条不同的路径，概率分别为 p₁=0.3 和 p₂=0.7，我们想求 p₁+p₂=1.0。

在 log 域：
```
log(p₁) ≈ -1.204
log(p₂) ≈ -0.357
```

**错误做法**：直接相加
```
log(p₁) + log(p₂) ≈ -1.204 + (-0.357) = -1.561
```
这实际上是在计算 log(p₁ × p₂)（乘法的 log），而不是 log(p₁ + p₂)！

如果我们错误地取 exp(-1.561) ≈ 0.21，这是完全错误的答案（应该是 1.0）。

**正确做法**：logaddexp
```
logaddexp(log(p₁), log(p₂)) = log(exp(log(p₁)) + exp(log(p₂)))
                             = log(0.3 + 0.7)
                             = log(1.0)
                             = 0
```

**直觉**：
- `log(a + b)` ≠ `log(a) + log(b)`，所以我们必须"还原"成概率，相加，再取 log
- numpy 的 `logaddexp` 就是为此设计的，它高效且数值稳定地实现了这个过程
- 三个或更多数相加可以嵌套调用：logaddexp(logaddexp(a, b), c)


In [ ]:
def log_softmax(logits, axis=-1):
    m = logits.max(axis=axis, keepdims=True)
    shifted = logits - m
    return shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))

In [ ]:
def ctc_forward(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    """CTC 前向算法：计算 log P(labels | log_probs)。

    Args:
        log_probs: (T, vocab_size) log-probability matrix.
        labels:    list of integer label ids (without blanks).
        blank:     blank token index.
                   注意：本笔记本 BLANK=2，请显式传入 blank=BLANK；
                   函数默认值 0 是 CTC 论文惯例，不适用于本例。

    Returns:
        log probability of the label sequence under CTC.
    """
    T, V = log_probs.shape
    L = len(labels)

    # 扩展标签：blank + label + blank + label + ... + blank
    lprime = [blank]
    for c in labels:
        lprime.append(c)
        lprime.append(blank)
    S = len(lprime)   # = 2L + 1

    NEG_INF = -1e30
    alpha = np.full((T, S), NEG_INF)

    # ✏️ 步骤1：初始条件
    # 第 0 帧只能从扩展标签的前两个位置出发：
    #   - 位置 0（初始 blank）
    #   - 位置 1（第一个字符）
    # 其他位置在第 0 帧无法到达（仍是 -inf）
    alpha[0, 0] = log_probs[0, lprime[0]]
    if S > 1:
        alpha[0, 1] = log_probs[0, lprime[1]]

    # ✏️ 步骤2：t=1..T-1 的递推
    # 对每一帧的每个位置，从三个可能的前驱状态来
    for t in range(1, T):
        for s in range(S):
            # 候选：停留在当前位置
            candidates = [alpha[t-1, s]]
            
            # 候选：从左边一格跳过来（向右推进 1 格）
            if s > 0:
                candidates.append(alpha[t-1, s-1])
            
            # 候选：从左边两格跳过来（向右推进 2 格）
            # 仅当当前位置不是 blank 且与两步前位置的标签不同时允许
            # 这个条件保证了路径折叠的正确性（不会把"A-blank-A"错当成"A"）
            if s > 1 and lprime[s] != lprime[s-2]:
                candidates.append(alpha[t-1, s-2])
            
            # 用 logaddexp 合并所有候选（在 log 域稳定相加）
            alpha[t, s] = np.logaddexp.reduce(candidates) + log_probs[t, lprime[s]]

    # ✏️ 步骤3：返回终态 log-sum
    # 最后一帧可停在两个位置：
    #   - 位置 S-1（最后一个字符，对应标签的最后字符）
    #   - 位置 S-2（末尾 blank，对应路径可以提前结束再用 blank 填充）
    # 取这两个状态的 logaddexp 作为最终答案
    return np.logaddexp(alpha[-1, -1], alpha[-1, -2])

## 半步练习 B · ctc_forward_brute_force（先写 / 先读此函数）

先把“求和的对象”弄明白，再写 DP。暴力枚举只在 T 很小时能跑，所以它是验证器，不是训练器。

In [ ]:
def ctc_forward_brute_force(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    """暴力枚举版 CTC（仅用于验证，T 较小时可用）。
    枚举所有长度为 T 的路径，保留折叠后等于 labels 的路径，用 logsumexp 聚合。
    """
    from itertools import product
    T, V = log_probs.shape

    def collapse(path):
        """去掉 blank、合并重复字符 → 等价于 CTC collapse。"""
        result, prev = [], None
        for c in path:
            if c == blank:
                prev = None
            elif c != prev:
                result.append(c)
                prev = c
        return result

    log_p_list = []
    for path in product(range(V), repeat=T):
        if collapse(list(path)) == list(labels):
            lp = sum(log_probs[t, path[t]] for t in range(T))
            log_p_list.append(lp)

    if not log_p_list:
        return -1e30

    arr = np.array(log_p_list)
    m = arr.max()
    return float(m + np.log(np.exp(arr - m).sum()))


# 快速自检（T=4 时暴力枚举可在 < 1 秒内完成）
np.random.seed(42)
_lp4 = log_softmax(np.random.randn(4, VOCAB_SIZE))
_ref = ctc_forward_brute_force(_lp4, LABELS, blank=BLANK)
print(f"暴力枚举参考值（T=4）: {_ref:.4f}  → 用于断言 DP 实现正确性")

In [ ]:
try:
    lp = ctc_forward(log_probs, LABELS, blank=BLANK)
    print(f'log P(labels | frames) = {lp:.4f}')
    print(f'P(labels | frames)     = {np.exp(lp):.8f}')
    assert np.isfinite(lp), '前向算法应返回有限值'
    assert lp < 0, 'log 概率应为负数'

    # 强验证：与暴力枚举参考实现对照（T=6 时 3^6=729 路径，可在 < 1 秒内枚举）
    lp_ref = ctc_forward_brute_force(log_probs, LABELS, blank=BLANK)
    print(f'暴力枚举参考值         = {lp_ref:.4f}')
    assert np.isclose(lp, lp_ref, atol=1e-5), \
        f'❌ 与参考值不符：DP={lp:.4f}, brute={lp_ref:.4f}，请检查递推或初始条件'

    print('✅ CTC 前向算法验证通过（与暴力枚举吻合至 1e-5）')
except (NotImplementedError, TypeError):
    print('⚠️ ctc_forward 尚未实现，请完成 TODO 步骤1-3')

## 复杂度与直觉

In [ ]:
try:
    # 展示：T 增大时计算量线性增长（而不是指数增长）
    print(f'{"T":>4}  {"S=2L+1":>6}  {"计算格数":>8}  {"log P":>8}')
    print('-' * 35)
    for T_test in [5, 10, 20, 50, 100]:
        _logits_t = np.random.randn(T_test, VOCAB_SIZE)
        _logits_t -= _logits_t.max(axis=1, keepdims=True)  # 数值稳定
        lp_test = _logits_t - np.log(np.exp(_logits_t).sum(axis=1, keepdims=True))
        result = ctc_forward(lp_test, LABELS, blank=BLANK)
        S = 2*len(LABELS)+1
        print(f'{T_test:>4}  {S:>6}  {T_test*S:>8}  {result:>8.3f}')
except (NotImplementedError, TypeError):
    print('⚠️ ctc_forward 尚未实现，请先完成 TODO 步骤1-3')

## 本课收束

| 概念 | 要记住的 |
|---|---|
| 扩展标签 l' | blank 插在每个标签之间，长度 2L+1 |
| 前向变量 α[t][s] | log 域递推，避免下溢 |
| 三个前驱 | 停留、跳 1、跳 2（限字符不同时）|
| 复杂度 | O(T·S) — 其中 `S = 2L + 1`，与暴力枚举路径的指数复杂度相比快得多 |
| L70 | Whisper 用 Attention 解码，但仍然是 token-by-token 的自回归过程 |

下一课（L70）从代码层面解剖 Whisper 架构：音频编码器（encoder）、交叉注意力（cross-attention）和文本解码器的完整实现。先把本课的 log 域 DP 和 L67 的填表思维串起来，再去看 Whisper。

## ✏️ 闭卷推导检查格 — CTC 前向算法 α 递推

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目 1 — α 递推公式**：CTC 前向变量 $\alpha(t, s)$ 满足递推：

$$\alpha(t, s) = \Big[\alpha(t-1, s) + \alpha(t-1, s-1) + \alpha(t-1, s-2) \cdot \mathbf{1}[\text{条件}]\Big] \cdot p(l'_s \mid x_t)$$

写出 $\mathbf{1}[\text{条件}]$ 的完整条件（涉及 blank 与相邻非 blank 标签的关系）。

**题目 2 — blank 合并规则**：

| 路径 | 折叠后标签 |
|------|-----------| 
| a - a | |
| a - ε - a | |
| a - ε - b | |

（ε = blank 符号）

**题目 3 — 扩展标签**：标签序列 $l = [a, b]$，写出 $l'$（含 blank）= ?

（在此处写推导...）

In [ ]:
# 验证：与预计算参考值做数值一致性对比（固定 seed=42）
import numpy as np

# 小规模例子：vocab={blank=0, a=1, b=2}，标签 l=[a,b]
# 扩展标签 l' = [ε, a, ε, b, ε]
T, vocab_size = 5, 3
np.random.seed(42)
log_probs_raw = np.random.dirichlet(np.ones(vocab_size), size=T)
log_probs = np.log(log_probs_raw + 1e-9)  # (T, vocab)

l_prime = [0, 1, 0, 2, 0]  # [blank, a, blank, b, blank]
S = len(l_prime)

# 前向递推（参考实现，不可修改）
NEG_INF = float('-inf')
log_alpha = np.full((T, S), NEG_INF)
log_alpha[0][0] = log_probs[0][l_prime[0]]
if S > 1: log_alpha[0][1] = log_probs[0][l_prime[1]]

for t in range(1, T):
    for s in range(S):
        paths = [log_alpha[t-1][s]]
        if s > 0:
            paths.append(log_alpha[t-1][s-1])
        # 第三路径：只在当前不是 blank 且与前一个非 blank 标签不同时才合法
        if s > 1 and l_prime[s] != 0 and l_prime[s] != l_prime[s-2]:
            paths.append(log_alpha[t-1][s-2])
        log_alpha[t][s] = np.logaddexp.reduce(paths) + log_probs[t][l_prime[s]]

# 最终 log P("ab" | 输入) = logaddexp(α(T-1, S-1), α(T-1, S-2))
log_p = np.logaddexp(log_alpha[-1][-1], log_alpha[-1][-2])

# 预计算参考值（seed=42 下固定）
REF_LOG_P = -1.361933   # 通过参考实现预先计算

assert not np.isnan(log_p), "log_p 是 NaN，递推可能有数值问题"
assert log_p < 0, "log 概率必须 < 0"
assert abs(log_p - REF_LOG_P) < 1e-4, (
    f"log_p = {log_p:.6f}，参考值 = {REF_LOG_P:.6f}\n"
    f"差值 = {abs(log_p - REF_LOG_P):.2e}（应 < 1e-4）\n"
    f"请检查题目 1 中 α 递推的三路径条件是否正确"
)
print(f"log P(ab | 输入) = {log_p:.6f}  参考值 = {REF_LOG_P:.6f}")
print("✅ CTC 前向算法验证通过（数值与参考一致）")

In [ ]:
# ── 手工验证：2帧 · 3词表（含blank=0）─────────────────────────────────
# 设定：vocab = {0:blank, 1:'a', 2:'b'}，标签序列 = [1]（即 'a'）
# 帧数 T=2，用极简概率矩阵手算前向概率
import numpy as np

# log 概率矩阵 shape=(T, V) — 用对称简单值便于手算
log_probs = np.log(np.array([
    [0.5, 0.4, 0.1],   # t=0: blank=0.5, a=0.4, b=0.1
    [0.3, 0.6, 0.1],   # t=1: blank=0.3, a=0.6, b=0.1
]))
labels = [1]   # target = 'a'

# 手算期望值（blank=0 默认）：
# 扩展标签 l' = [0(blank), 1(a), 0(blank)]
# 有效路径（折叠后等于 [a]）：
#   blank→a (0,1): 0.5×0.6 = 0.30
#   a→blank (1,0): 0.4×0.3 = 0.12
#   a→a     (1,1): 0.4×0.6 = 0.24  ← 连续相同字符折叠合并，合法路径
# P(labels|probs) = 0.30 + 0.12 + 0.24 = 0.66
expected_log_p = np.log(0.66)

try:
    # 调用本课实现的 ctc_forward（blank 默认值 0，与本例一致）
    log_p = ctc_forward(log_probs, labels)   # blank=0 (default)

    np.testing.assert_allclose(log_p, expected_log_p, atol=0.05,
        err_msg=f"CTC 前向概率与手算值不符：得到 {np.exp(log_p):.4f}，期望 ≈ 0.66")
    print(f"✅ CTC 手工验证通过：P(a|frame) = {np.exp(log_p):.4f}（期望 ≈ 0.66）")
    print("   有效路径：blank→a(0.30)，a→blank(0.12)，a→a(0.24)（连续同字符可折叠）")
except (NotImplementedError, TypeError):
    print('⚠️ ctc_forward 尚未实现，请完成 TODO 步骤1-3 后再运行本手工验证')

## 理解路径折叠与重复计数

**为什么路径中可以有连续相同字符（比如"aa"），但"a-blank-a"却被特殊处理？**

考虑目标标签是 `[a]`（单个字符）的情况：

```
扩展标签 l' = [∅, a, ∅]（位置 0, 1, 2）

可能的路径及其折叠结果：
─────────────────────────────────────────────
路径（每帧选择一个位置）       折叠后        是否合法
─────────────────────────────────────────────
∅ → ∅ → ∅                    []           否（没有字符）
∅ → ∅ → a                    [a]          ✓
∅ → a → ∅                    [a]          ✓
∅ → a → a                    [a]          ✓（aa折叠成a）
a → ∅ → ∅                    [a]          ✓
a → ∅ → a                    [a]          ✓（a-blank-a折叠成a）
a → a → ∅                    [a]          ✓（aa折叠成a）
a → a → a                    [a]          ✓（aaa折叠成a）
─────────────────────────────────────────────
```

**关键观察**：
1. **连续相同字符（a → a）是合法的**，折叠后仍然是 `[a]`
2. **中间有 blank 的也是合法的（a → ∅ → a）**，折叠规则是"去掉 blank、合并相邻重复"，所以也变成 `[a]`

**那么为什么要加条件 `l'[s] ≠ l'[s-2]`？**

这个条件的作用是**在 DP 中避免重复计算这些合法路径**。

比如，考虑目标 `[a, a]`（两个 a）的情况：
```
扩展标签 l' = [∅, a, ∅, a, ∅]（位置 0, 1, 2, 3, 4）
```

现在考虑 t=2 时计算 α[2][3]（第 3 个位置，即第二个 a）。前驱可能来自：
- α[1][3]（停留）
- α[1][2]（跳 1 步）
- **α[1][1]（跳 2 步）？**

如果我们**错误地允许 α[1][1] → α[2][3]**（从第一个 a 直接跳到第二个 a），路径就是：
```
... → a → a    (路径在第1帧和第2帧都选位置1，即都是'a'）
```
折叠后就是 `[a]`（不是目标的 `[a, a]`）！

而我们**已经通过** α[1][2] → α[2][3]（从中间的 blank 跳到第二个 a）把这种情况计数了：
```
... → ∅ → a    (路径在第1帧选位置2（blank），第2帧选位置3（a））
```
折叠后也是包含两个 a 的路径。

所以条件 `l'[s] ≠ l'[s-2]` 的作用是：**禁止从位置 s-2 直接跳到位置 s，当这两个位置都是同一个字符时**。这样可以避免在 DP 过程中重复计数那些应该通过"停留"或"跳一步"来覆盖的路径。

总结：
- 目标标签可以有重复字符（a 后跟 a）
- 路径可以有各种组合（包括连续相同、中间 blank 等）
- 但在 DP 递推时，我们通过"禁止某些跳过方式"来确保每条路径只被计数一次


In [ ]:
# ✏️ 本课自评
l69_review = {
    "exp_path_count_understood": None,  # 理解朴素枚举为何是指数级（V^T条路径）？True/False
    "expanded_label_l_prime":    None,  # 记住l'=[∅,a,∅,b,∅]，长度S=2L+1？True/False
    "forward_var_recursion":     None,  # 能默写α[t][s]的递推公式（情况1/2/初始化）？True/False
    "ctc_forward_impl":          None,  # ctc_forward()实现正确，与暴力枚举误差<1e-5？True/False
    "whiteboard_passed":         None,  # 白板挑战α递推推导通关？True/False
}

unfilled = [k for k, v in l69_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l69_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L69 全部通关！进入 L70：Whisper 架构解析')

---

→ **下一课**　[L70 · Whisper 架构解析](L70_whisper_arch.ipynb)

> 下节课将学习 **Whisper 架构解析**：Log-Mel 输入、Transformer Encoder-Decoder、多任务头。先把本课的 log 域 DP 和 L67 的填表思维串起来，再去读 Whisper。